# Fine-Tuning Pipeline
**ContextFlow AI** - Supervised Fine-Tuning dengan Unsloth + LoRA

1. Load preprocessed dataset
2. Load base model (Qwen2.5-3B-Instruct via Unsloth)
3. Konfigurasi LoRA
4. Training dengan SFTTrainer
5. Save model fine-tuned

**Notebook ini terhubung langsung dengan modul Python di `app/`:**
- `app.training.trainer` — Training dengan Unsloth
- `app.preprocessing.pipeline` — Preprocessing data
- `app.utils.config` — Konfigurasi
- `app.utils.logger` — Logging

In [5]:
import sys
import os

# Pastikan root project ada di sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
from datasets import load_from_disk
from unsloth import FastLanguageModel
from app.utils.config import config
from app.utils.logger import logger

# Import fungsi dari app/training/trainer.py
from app.training.trainer import (
    load_model_and_tokenizer,
    apply_lora,
    get_training_args,
    run_training,
)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print(f'Base model: {config.BASE_MODEL}')
print(f'Max seq length: {config.MAX_SEQ_LENGTH}')

PyTorch: 2.7.1+cu118
CUDA available: True
GPU: NVIDIA GeForce RTX 2050
VRAM: 4.0 GB
Base model: unsloth/Qwen2.5-3B-Instruct-bnb-4bit
Max seq length: 512


## 1. Load Preprocessed Data

In [6]:
train_ds = load_from_disk(os.path.join(config.FORMATTED_DATA_DIR, 'train'))
val_ds = load_from_disk(os.path.join(config.FORMATTED_DATA_DIR, 'val'))

print(f'Train: {len(train_ds)} samples')
print(f'Val: {len(val_ds)} samples')
print(f'Columns: {train_ds.column_names}')
print()
print('Sample text:')
print(train_ds[0]['text'][:500])

Train: 41866 samples
Val: 4652 samples
Columns: ['instruction', 'input', 'output', 'text', 'source']

Sample text:
### Instruction:
When did Virgin Australia start operating?

### Input:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to d


## 2. Load Model & Tokenizer (Unsloth)

Menggunakan `load_model_and_tokenizer()` dari `app/training/trainer.py`.
Model di-load dengan **4-bit quantization** via Unsloth `FastLanguageModel`.

In [7]:
# Load model & tokenizer dari app/training/trainer.py
model, tokenizer = load_model_and_tokenizer(config.BASE_MODEL)

print(f'Vocab size: {len(tokenizer)}')
print(f'Pad token: {tokenizer.pad_token}')
print(f'Model dtype: {model.dtype}')

sample = train_ds[0]['text']
tokens = tokenizer(sample, truncation=True, max_length=config.MAX_SEQ_LENGTH)
print(f'Sample token length: {len(tokens["input_ids"])}')

2026-05-28 16:15:41 | INFO | app.training.trainer:32 - Loading model: unsloth/Qwen2.5-3B-Instruct-bnb-4bit (Unsloth 4-bit)


NotImplementedError: Unsloth: unsloth/Qwen2.5-3B-Instruct-bnb-4bit is not supported in your current Unsloth version! Please update Unsloth via:

pip uninstall unsloth -y
pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 3. Konfigurasi LoRA (Unsloth)

Menggunakan `apply_lora()` dari `app/training/trainer.py`.
- r=16, alpha=32
- Target: attention + MLP modules (7 total)
- Gradient checkpointing: `"unsloth"` (30% lebih hemat VRAM)

In [ ]:
# Apply LoRA dari app/training/trainer.py
model = apply_lora(model)

## 4. Training Arguments

Menggunakan `get_training_args()` dari `app/training/trainer.py`.
- Auto-detect fp16/bf16
- Optimizer: `adamw_8bit` (hemat VRAM)

In [ ]:
output_dir = config.OUTPUT_MODEL_DIR

# Training args dari app/training/trainer.py
training_args = get_training_args(output_dir)

print(f'Epochs: {training_args.num_train_epochs}')
print(f'Batch size: {training_args.per_device_train_batch_size}')
print(f'Learning rate: {training_args.learning_rate}')
print(f'Optimizer: {training_args.optim}')
print(f'fp16: {training_args.fp16} | bf16: {training_args.bf16}')

## 5. Fine-Tuning

In [ ]:
from trl import SFTTrainer
from transformers import EarlyStoppingCallback

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=config.MAX_SEQ_LENGTH,
    packing=False,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print('Trainer ready!')

In [ ]:
# Mulai training
trainer.train()

## 6. Save Model

In [ ]:
# Save LoRA adapter
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f'Model saved to {output_dir}')
for f in os.listdir(output_dir):
    filepath = os.path.join(output_dir, f)
    if not os.path.isdir(filepath):
        size = os.path.getsize(filepath)
        print(f'  {f}: {size/1024/1024:.1f} MB')

## 7. Quick Inference Test

In [ ]:
# Set model ke mode inference (Unsloth — 2x lebih cepat)
FastLanguageModel.for_inference(model)

# Prompt ChatML (Qwen2.5) — konsisten dengan formatter.py
prompt = '<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nApa prosedur pengajuan cuti tahunan?\nKaryawan harus mengisi form HR-01.<|im_end|>\n<|im_start|>assistant\n'
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=256, temperature=0.7, do_sample=True, top_p=0.9)

result = tokenizer.decode(output[0], skip_special_tokens=True)
print('Generated:')
# Extract response setelah assistant
if 'assistant' in result:
    print(result.split('assistant')[-1].strip())
else:
    print(result[len(prompt):])

## 8. Training Log & Metrics

In [ ]:
# Tampilkan training metrics
if hasattr(trainer, 'state') and trainer.state.log_history:
    import pandas as pd
    import matplotlib.pyplot as plt

    logs_df = pd.DataFrame(trainer.state.log_history)
    print('Training Logs:')
    display(logs_df)

    # Plot loss
    train_logs = logs_df[logs_df['loss'].notna()] if 'loss' in logs_df.columns else None
    eval_logs = logs_df[logs_df['eval_loss'].notna()] if 'eval_loss' in logs_df.columns else None

    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    if train_logs is not None and len(train_logs) > 0:
        ax.plot(train_logs['step'], train_logs['loss'], label='Train Loss', marker='o')
    if eval_logs is not None and len(eval_logs) > 0:
        ax.plot(eval_logs['step'], eval_logs['eval_loss'], label='Eval Loss', marker='s')
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.set_title('Training & Evaluation Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Tidak ada training logs.')